In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import mlflow

In [0]:
df = spark.table("fmcg.gold.fact_sales_features")

display(df.limit(10))

product_code,channel,date,sold_quantity,division,category,product,price_inr,market,platform,lag_1,lag_2,lag_3,lag_12,rolling_mean_3,month,quarter,year
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-10-01,75013,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar,80900,75681,76219,null,77600.0,10,4,2025
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-11-01,71409,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar,75013,80900,75681,null,77198.0,11,4,2025
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-12-01,9061,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar,71409,75013,80900,null,75774.0,12,4,2025
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Acquisition,2025-10-01,114977,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),100.0,India,Sports Bar,121121,120725,124782,null,122209.33333333333,10,4,2025
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Acquisition,2025-11-01,122116,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),100.0,India,Sports Bar,114977,121121,120725,null,118941.0,11,4,2025
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Acquisition,2025-12-01,19981,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),100.0,India,Sports Bar,122116,114977,121121,null,119404.66666666667,12,4,2025
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,Acquisition,2025-10-01,170505,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (25g),86.0,India,Sports Bar,163271,155582,155316,null,158056.33333333334,10,4,2025
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,Acquisition,2025-11-01,176859,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (25g),86.0,India,Sports Bar,170505,163271,155582,null,163119.33333333334,11,4,2025
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,Acquisition,2025-12-01,25733,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (25g),86.0,India,Sports Bar,176859,170505,163271,null,170211.66666666666,12,4,2025
2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896,Acquisition,2025-10-01,120413,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (45g),108.0,India,Sports Bar,116405,131111,121823,null,123113.0,10,4,2025


In [0]:
last_date = (
    df
    .select(F.max("date"))
    .collect()[0][0]
)

print("Latest historical date:", last_date)  # The latest date

Latest historical date: 2025-12-01


In [0]:
# generate the next 12 months
future_dates = pd.date_range(
    start=pd.Timestamp(last_date) + pd.DateOffset(months=1),
    periods=12,
    freq="MS"
)

print(future_dates)

DatetimeIndex(['2026-01-01', '2026-02-01', '2026-03-01', '2026-04-01',
               '2026-05-01', '2026-06-01', '2026-07-01', '2026-08-01',
               '2026-09-01', '2026-10-01', '2026-11-01', '2026-12-01'],
              dtype='datetime64[ns]', freq='MS')


In [0]:
# Loading the best model
model = mlflow.lightgbm.load_model("models:/fmcg.gold.sales_forecast_model@champion_model")

In [0]:
# preparing historical data 
hist = (
    df
    .select(
        "product_code",
        "channel",
        "date",
        "sold_quantity",
        "price_inr"
    )
    .toPandas()
)

hist["date"] = pd.to_datetime(hist["date"])

hist = hist.sort_values(
    ["product_code", "channel", "date"]
).reset_index(drop=True)

In [0]:
forecast_rows = []

for future_date in future_dates:

    rows = []

    for (product_code, channel), group in hist.groupby(
        ["product_code", "channel"]
    ):

        group = group.sort_values("date")

        last_values = group["sold_quantity"].tail(3).tolist()

        if len(last_values) < 3:
            continue

        lag_1 = last_values[-1]
        lag_2 = last_values[-2]
        lag_3 = last_values[-3]

        price = group["price_inr"].iloc[-1]

        rows.append({
            "product_code": product_code,
            "channel": channel,
            "date": future_date,

            "lag_1": lag_1,
            "lag_2": lag_2,
            "lag_3": lag_3,

            "month": future_date.month,
            "quarter": future_date.quarter,

            "price_inr": price
        })

    future_df = pd.DataFrame(rows)

    future_df["product_code"] = (
        future_df["product_code"].astype("category")
    )

    future_df["channel"] = (
        future_df["channel"].astype("category")
    )

    X_future = future_df[
        [
            "lag_1",
            "lag_2",
            "lag_3",
            "month",
            "quarter",
            "price_inr",
            "product_code",
            "channel"
        ]
    ]

    predictions = model.predict(X_future)

    predictions = np.clip(predictions, 0, None)

    future_df["forecast_quantity"] = predictions

    forecast_rows.append(future_df)

    # Feed predictions back into history
    new_history = future_df[
        [
            "product_code",
            "channel",
            "date",
            "forecast_quantity",
            "price_inr"
        ]
    ].rename(
        columns={
            "forecast_quantity": "sold_quantity"
        }
    )

    hist = pd.concat(
        [hist, new_history],
        ignore_index=True
    )

In [0]:
forecast_pd = pd.concat(
    forecast_rows,
    ignore_index=True
)

display(
    forecast_pd.sort_values(
        ["product_code", "channel", "date"]
    ).head(30)
)

product_code,channel,date,lag_1,lag_2,lag_3,month,quarter,price_inr,forecast_quantity
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2026-01-01T00:00:00.000Z,9061.0,71409.0,75013.0,1,1,281.0,4247.216488715052
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2026-02-01T00:00:00.000Z,4247.216488715052,9061.0,71409.0,2,1,281.0,2968.5196558548814
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2026-03-01T00:00:00.000Z,2968.5196558548814,4247.216488715052,9061.0,3,1,281.0,2277.3065608123543
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2026-04-01T00:00:00.000Z,2277.3065608123543,2968.5196558548814,4247.216488715052,4,2,281.0,1181.193909733644
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2026-05-01T00:00:00.000Z,1181.193909733644,2277.3065608123543,2968.5196558548814,5,2,281.0,1683.3880929587776
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2026-06-01T00:00:00.000Z,1683.3880929587776,1181.193909733644,2277.3065608123543,6,2,281.0,1452.9004623434105
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2026-07-01T00:00:00.000Z,1452.9004623434105,1683.3880929587776,1181.193909733644,7,3,281.0,1399.2994054522458
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2026-08-01T00:00:00.000Z,1399.2994054522458,1452.9004623434105,1683.3880929587776,8,3,281.0,1387.2379169130563
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2026-09-01T00:00:00.000Z,1387.2379169130563,1399.2994054522458,1452.9004623434105,9,3,281.0,1748.3082727345338
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2026-10-01T00:00:00.000Z,1748.3082727345338,1387.2379169130563,1399.2994054522458,10,4,281.0,1784.6061622263255


In [0]:
# adding the metadata 
forecast_pd["forecast_run_date"] = pd.Timestamp.now()

In [0]:
forecast_spark = spark.createDataFrame(forecast_pd)

In [0]:
forecast_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "fmcg.gold.fact_sales_forecast"
    )

In [0]:
display(
    spark.table(
        "fmcg.gold.fact_sales_forecast"
    )
)

product_code,channel,date,lag_1,lag_2,lag_3,month,quarter,price_inr,forecast_quantity,forecast_run_date
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2026-04-01T00:00:00.000Z,2277.3065608123543,2968.5196558548814,4247.216488715052,4,2,281.0,1181.193909733644,2026-08-28T15:30:24.405Z
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Acquisition,2026-04-01T00:00:00.000Z,3426.427061301314,2401.874373164253,3890.185692338248,4,2,100.0,916.3731101905156,2026-08-28T15:30:24.405Z
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,Acquisition,2026-04-01T00:00:00.000Z,3426.427061301314,2401.874373164253,3890.185692338248,4,2,86.0,916.3731101905156,2026-08-28T15:30:24.405Z
2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896,Acquisition,2026-04-01T00:00:00.000Z,3426.427061301314,2401.874373164253,3890.185692338248,4,2,108.0,916.3731101905156,2026-08-28T15:30:24.405Z
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,Acquisition,2026-04-01T00:00:00.000Z,815.3102121471128,2799.6885597961987,2442.4174465208503,4,2,493.0,1665.9864456522607,2026-08-28T15:30:24.405Z
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,Acquisition,2026-04-01T00:00:00.000Z,1442.0145593009402,2265.6625525913296,3391.2448974818562,4,2,187.0,1012.3729036851495,2026-08-28T15:30:24.405Z
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,Acquisition,2026-04-01T00:00:00.000Z,1981.8575545720826,2625.8625789038315,3552.9718909475046,4,2,440.0,921.9597567877214,2026-08-28T15:30:24.405Z
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,Acquisition,2026-04-01T00:00:00.000Z,1442.0145593009402,2265.6625525913296,3403.4924706227966,4,2,296.0,1012.3729036851495,2026-08-28T15:30:24.405Z
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,Acquisition,2026-04-01T00:00:00.000Z,3426.427061301314,2401.874373164253,3890.185692338248,4,2,50.0,916.3731101905156,2026-08-28T15:30:24.405Z
889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2,Acquisition,2026-04-01T00:00:00.000Z,3426.427061301314,2401.874373164253,3890.185692338248,4,2,138.0,916.3731101905156,2026-08-28T15:30:24.405Z
